In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

n_samples = 40
n_features = 20

X = rng.normal(size=(n_samples, n_features))

true_w = np.zeros(n_features)
true_w[:3] = [5, -3, 2]

noise = rng.normal(0, 1.5, size=n_samples)

y = X @ true_w + noise

print("True coefficients:")
print(true_w)

True coefficients:
[ 5. -3.  2.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.]


In [2]:
linear = LinearRegression()
linear.fit(X, y)

print("LinearRegression coefficients:")
print(np.round(linear.coef_, 3))

LinearRegression coefficients:
[ 5.436 -2.798  2.015 -0.251  0.627 -0.205  0.669 -0.45  -0.177 -0.055
  0.015  0.195 -0.627 -0.265  0.102 -0.257  0.302 -0.427  0.359 -0.487]


In [3]:
ridge = Ridge(alpha=10)
ridge.fit(X, y)

print("Ridge coefficients:")
print(np.round(ridge.coef_, 3))

Ridge coefficients:
[ 3.732 -1.74   1.543  0.08  -0.067 -0.037  0.364  0.422 -0.122  0.641
  0.33   0.281 -0.382 -0.125 -0.11  -0.395  0.072 -0.191  0.382  0.13 ]


In [4]:
lasso = Lasso(alpha=0.2, max_iter=10000)
lasso.fit(X, y)

print("Lasso coefficients:")
print(np.round(lasso.coef_, 3))

Lasso coefficients:
[ 5.05  -2.417  1.978 -0.     0.037 -0.     0.     0.    -0.     0.
  0.     0.206 -0.067 -0.    -0.    -0.04   0.    -0.     0.     0.   ]


In [5]:
for alpha in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X, y)

    print(
        f"Ridge alpha={alpha:<6} "
        f"coef norm={np.linalg.norm(ridge.coef_):.3f}"
    )

print()

for alpha in [0.01, 0.05, 0.1, 0.2, 0.5, 1]:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X, y)

    zero_count = np.sum(np.isclose(lasso.coef_, 0))

    print(
        f"Lasso alpha={alpha:<5} "
        f"zero coefficients={zero_count}/20"
    )

Ridge alpha=0.01   coef norm=6.616
Ridge alpha=0.1    coef norm=6.568
Ridge alpha=1      coef norm=6.190
Ridge alpha=10     coef norm=4.561
Ridge alpha=100    coef norm=1.712

Lasso alpha=0.01  zero coefficients=1/20
Lasso alpha=0.05  zero coefficients=7/20
Lasso alpha=0.1   zero coefficients=11/20
Lasso alpha=0.2   zero coefficients=13/20
Lasso alpha=0.5   zero coefficients=17/20
Lasso alpha=1     zero coefficients=17/20


In [6]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parents[1]

df = pd.read_csv(ROOT / "data" / "duolingo_flagship_v5.csv")
split_df = pd.read_csv(ROOT / "data" / "split_users.csv")

cv_users = set(
    split_df.loc[split_df["split"] == "cv", "user_id"]
)

df_cv = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features]
y = df_cv["p_recall"]
groups = df_cv["user_id"]

print(X.shape)
print(groups.nunique())

(14438, 5)
2125


In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

gkf = GroupKFold(n_splits=5)

ridge_alphas = [0.1, 1, 10, 100]

for alpha in ridge_alphas:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha))
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        fold_rmses.append(rmse)

    print(
        f"alpha={alpha:<5} "
        f"RMSE={np.mean(fold_rmses):.6f} ± {np.std(fold_rmses):.6f}"
    )

alpha=0.1   RMSE=0.273695 ± 0.009840
alpha=1     RMSE=0.273656 ± 0.009891
alpha=10    RMSE=0.273498 ± 0.010159
alpha=100   RMSE=0.273508 ± 0.010322


In [8]:
from sklearn.linear_model import Lasso

lasso_alphas = [0.0001, 0.001, 0.01, 0.1]

for alpha in lasso_alphas:
    fold_rmses = []
    zero_counts = []

    for train_idx, val_idx in gkf.split(X, y, groups=groups):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Lasso(alpha=alpha, max_iter=10000))
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        fold_rmses.append(rmse)

        coef = pipeline.named_steps["model"].coef_
        zero_counts.append(np.sum(np.isclose(coef, 0)))

    print(
        f"alpha={alpha:<7} "
        f"RMSE={np.mean(fold_rmses):.6f} ± {np.std(fold_rmses):.6f} "
        f"avg zeros={np.mean(zero_counts):.1f}/5"
    )

alpha=0.0001  RMSE=0.273533 ± 0.010116 avg zeros=0.0/5
alpha=0.001   RMSE=0.273561 ± 0.010338 avg zeros=1.4/5
alpha=0.01    RMSE=0.273919 ± 0.010266 avg zeros=3.0/5
alpha=0.1     RMSE=0.275398 ± 0.010158 avg zeros=5.0/5


## Part 10 Conclusion

### Ridge vs Lasso

Ridge regression adds an L2 penalty:

$$
J_{\text{ridge}}
=
J(w)
+
\alpha \sum_j w_j^2
$$

Lasso regression adds an L1 penalty:

$$
J_{\text{lasso}}
=
J(w)
+
\alpha \sum_j |w_j|
$$

Ridge mainly performs coefficient shrinkage, while Lasso can also force some coefficients exactly to zero and therefore produce sparse models.

### Probabilistic Interpretation

A zero-centered Gaussian prior on the weights,

$$
w_j \sim \mathcal N(0,\tau^2)
$$

produces an L2 penalty under MAP estimation:

$$
\text{Gaussian likelihood}
+
\text{Gaussian prior}
\rightarrow
\text{MAP}
\rightarrow
\text{Ridge}
$$

A zero-centered Laplace prior produces an L1 penalty:

$$
\text{Laplace prior}
\rightarrow
L1
\rightarrow
\text{Lasso}
$$

Regularization can therefore be interpreted as adding a prior preference about plausible parameter values rather than simply weakening the model arbitrarily.

A smaller prior variance corresponds to a stronger belief that weights should remain close to zero, which results in stronger regularization.

### Synthetic Regression Result

The synthetic experiment used 20 features while only the first 3 contained real signal.

Linear regression assigned non-zero coefficients to many noise features.

Ridge reduced the overall magnitude of the coefficient vector as `alpha` increased.

Lasso increasingly produced exact-zero coefficients as `alpha` increased:

- `alpha=0.01` → `1/20` zero coefficients
- `alpha=0.10` → `11/20`
- `alpha=0.20` → `13/20`
- `alpha=0.50` → `17/20`

This directly demonstrated the difference between shrinkage and sparsity.

### Flagship Result

Using 5-fold GroupKFold on unseen users:

| Model | CV RMSE |
|---|---:|
| Linear Regression | `0.273699 ± 0.009834` |
| Ridge, alpha=10 | `0.273498 ± 0.010159` |
| Lasso, alpha=0.0001 | `0.273533 ± 0.010116` |
| Lasso, alpha=0.1 | `0.275398 ± 0.010158` |

Moderate Ridge regularization produced a small improvement over ordinary linear regression.

The improvement is too small to declare Ridge definitively better at this stage, so systematic hyperparameter selection is deferred to Part 12.

For Lasso, stronger regularization increased sparsity. At `alpha=0.1`, all five feature coefficients became zero, reducing the model to an intercept-only predictor:

$$
\hat y = \bar y_{\text{train}}
$$

As a result, its CV score became identical to the mean baseline.

The main lesson is that stronger regularization or greater sparsity does not automatically imply better generalization. Excessive regularization can remove useful signal and cause underfitting.